# Profiling of the raw HR data (Group 2)

**Goal:** find and count every problem in the raw files, then compare our counts with `injected_errors.json` (the answer key).

**Rules**
- Read only from `data/raw/` (the data lake). Never save over those files.
- Cleaning comes later. This notebook only *measures* the problems and records our decisions.


## 0. Setup

In [1]:

import json
import os
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

RUN_DATE = "2026-11-03"  # Must match your date folder name

# Find the project root automatically
# Notebook location: project_root/src/notebooks/
ROOT = Path.cwd().parent

print("Working folder:", Path.cwd())
print("Project folder:", ROOT)
print("Data folder exists:", (ROOT / "data").exists())

# Define raw data folder
RAW = ROOT / "data" / "raw"

print("\nFiles found in the data lake:")
for p in sorted(RAW.rglob("*.*")):
    print("  ", p.relative_to(ROOT))

# Read everything as TEXT so nothing is converted or hidden.
READ = dict(dtype=str, keep_default_na=False, na_values=[""])

# Define file paths
emp_path = RAW / "employees" / RUN_DATE / "employees.xlsx"
att_path = RAW / "attendance" / RUN_DATE / "attendance.csv"
perf_path = RAW / "performance" / RUN_DATE / "performance.csv"
dept_path = ROOT / "data" / "reference" / "departments.csv"
key_path = ROOT / "data" / "source" / "injected_errors.json"

# Check whether all required files exist
required_files = [
    emp_path,
    att_path,
    perf_path,
    dept_path,
    key_path
]

missing_files = [p for p in required_files if not p.exists()]

if missing_files:
    print("\nMissing files:")
    for p in missing_files:
        print("  ", p)
    raise FileNotFoundError("Please check the missing file paths above.")

# Load datasets
emp = pd.read_excel(emp_path, **READ)
att = pd.read_csv(att_path, **READ)
perf = pd.read_csv(perf_path, **READ)
dept = pd.read_csv(dept_path, **READ)

# Load answer key
with open(key_path, encoding="utf-8") as f:
    key = json.load(f)

# Store our detected error counts
found = {
    "employees": {},
    "attendance": {},
    "performance": {}
}

def parse_dates(s):
    """Return (ISO-parsed, dd/mm/yyyy-parsed).
    A value neither can read is NaT in both.
    """
    iso = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    dmy = pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")
    return iso, dmy

# Display dataset dimensions
print("\nemployees :", emp.shape)
print("attendance:", att.shape)
print("performance:", perf.shape)
print("departments:", dept.shape)


Working folder: d:\projects\Group-2-Data-Engineering-Capstone-Project\notebooks
Project folder: d:\projects\Group-2-Data-Engineering-Capstone-Project
Data folder exists: True

Files found in the data lake:
   data\raw\attendance\2026-11-03\attendance.csv
   data\raw\employees\2026-11-03\employees.xlsx
   data\raw\performance\2026-11-03\performance.csv

employees : (206, 6)
attendance: (13040, 7)
performance: (410, 5)
departments: (7, 2)


## 1. Employees (one row = one person)

### 1.1 Structure

In [2]:
print("Rows and columns:", emp.shape)
print("Columns:", list(emp.columns))
emp.head(10)

Rows and columns: (206, 6)
Columns: ['employee_id', 'department', 'job_title', 'hire_date', 'employment_status', 'salary_band']


,employee_id,department,job_title,hire_date,employment_status,salary_band
0,E0001,Operations,Operations Officer,2026-04-26,Active,B2
1,E0002,Finance,Accountant,2025-08-03,Active,B1
2,E0003,Ops,Operations Officer,2018-05-03,Active,B2
3,E0024,Marketing,Content Creator,17/02/2018,Active,B1
4,E0004,Marketing,Marketing Executive,2024-04-17,Active,B4
5,E0005,IT,IT Support,2018-01-27,Active,B2
6,E0006,Operations,Operations Manager,2021-02-12,Active,B3
7,E0007,Finance,Finance Analyst,2019-02-01,Active,B3
8,E0008,Operations,Operations Officer,2026-03-08,Active,B1
9,E0009,Customer Service,Team Leader,2018-11-19,Active,B3


### 1.2 Missing values

In [3]:
print(emp.isna().sum())
found["employees"]["missing_department"] = int(emp["department"].isna().sum())
found["employees"]["missing_job_title"] = int(emp["job_title"].isna().sum())

employee_id          0
department           5
job_title            4
hire_date            0
employment_status    0
salary_band          0
dtype: int64


### 1.3 Duplicates
Two different questions: is the **employee_id** repeated, and is the **whole row** repeated?

In [4]:
rep_ids = int(emp.duplicated("employee_id").sum())
rep_rows = int(emp.duplicated().sum())
print("Repeated employee_id (extra copies):", rep_ids)
print("Fully identical rows               :", rep_rows)
print("Conflicts (same ID, something differs):", rep_ids - rep_rows)
found["employees"]["duplicate_employee_id_rows"] = rep_ids

emp[emp.duplicated("employee_id", keep=False)].sort_values("employee_id")

Repeated employee_id (extra copies): 6
Fully identical rows               : 3
Conflicts (same ID, something differs): 3


,employee_id,department,job_title,hire_date,employment_status,salary_band
3,E0024,Marketing,Content Creator,17/02/2018,Active,B1
24,E0024,Marketing,Marketing Manager,17/02/2018,Active,B1
31,E0067,Operations,Logistics Coordinator,2019-10-24,Active,B4
68,E0067,Operations,Operations Manager,2019-10-24,Active,B4
116,E0113,Operations,Logistics Coordinator,2018-04-22,Active,B4
172,E0113,Operations,Operations Manager,2018-04-22,Active,B4
95,E0142,Finance,Accountant,2021-07-16,Active,B2
145,E0142,Finance,Accountant,2021-07-16,Active,B2
104,E0173,Sales,Sales Executive,2025-05-17,Active,B4
177,E0173,Sales,Sales Executive,2025-05-17,Active,B4


### 1.4 Department labels
Every department must exist in the official list (`departments.csv`).

In [5]:
official = set(dept["department"])
bad_dept = emp[emp["department"].notna() & ~emp["department"].isin(official)]
print("Rows with a department not in the official list:", len(bad_dept))
found["employees"]["inconsistent_department_labels"] = len(bad_dept)
bad_dept["department"].value_counts()

Rows with a department not in the official list: 25


department
Ops                       4
I.T                       3
Finance Dept              3
Information Technology    2
sales                     2
CS                        2
SALES                     2
finance                   1
IT                        1
it                        1
Cust. Service             1
FINANCE                   1
Mktg                      1
H.R.                      1
Name: count, dtype: int64

### 1.5 Job title casing

In [6]:
t = emp["job_title"].dropna()
odd_case = t[(t == t.str.lower()) | (t == t.str.upper())]
print("Job titles written all lowercase or all UPPERCASE:", len(odd_case))
found["employees"]["job_title_casing_issues"] = len(odd_case)
odd_case.value_counts()

Job titles written all lowercase or all UPPERCASE: 8


job_title
logistics coordinator     1
IT SUPPORT                1
customer service agent    1
SALES EXECUTIVE           1
MARKETING MANAGER         1
hr manager                1
support specialist        1
IT MANAGER                1
Name: count, dtype: int64

### 1.6 Hire dates

In [7]:
iso, dmy = parse_dates(emp["hire_date"])
non_iso = iso.isna() & dmy.notna()
unreadable = iso.isna() & dmy.isna() & emp["hire_date"].notna()
parsed = iso.fillna(dmy)
future = parsed > pd.Timestamp(RUN_DATE)

print("ISO format (YYYY-MM-DD)  :", int(iso.notna().sum()))
print("Other format (dd/mm/yyyy):", int(non_iso.sum()))
print("Unreadable dates         :", int(unreadable.sum()))
print("Hire date in the future  :", int(future.sum()))
found["employees"]["non_iso_hire_date_format"] = int(non_iso.sum())
found["employees"]["future_hire_date"] = int(future.sum())

emp[future | non_iso][["employee_id", "hire_date"]].head(15)

ISO format (YYYY-MM-DD)  : 195
Other format (dd/mm/yyyy): 11
Unreadable dates         : 0
Hire date in the future  : 3


,employee_id,hire_date
3,E0024,17/02/2018
24,E0024,17/02/2018
36,E0035,14/07/2025
43,E0042,2027-09-29
47,E0046,06/02/2019
53,E0052,25/11/2018
74,E0073,16/08/2022
86,E0085,27/01/2026
99,E0097,08/10/2022
112,E0109,15/02/2024


## 2. Attendance (one row = one employee on one working day)

### 2.1 Structure, missing values, duplicates

In [8]:
print("Rows and columns:", att.shape)
print()
print("Missing values per column:")
print(att.isna().sum())
print()
dup_rows = int(att.duplicated().sum())
print("Fully identical rows      :", dup_rows)
print("Repeated attendance_id    :", int(att.duplicated("attendance_id").sum()))
found["attendance"]["exact_duplicate_rows"] = dup_rows
att.head()

Rows and columns: (13040, 7)

Missing values per column:
attendance_id         0
employee_id           0
attendance_date       0
status                0
check_in           1027
check_out           996
hours_worked          0
dtype: int64

Fully identical rows      : 40
Repeated attendance_id    : 40


,attendance_id,employee_id,attendance_date,status,check_in,check_out,hours_worked
0,A000001,E0001,2026-08-03,Present,07:48,17:40,9.87
1,A000002,E0001,2026-08-04,Absent,NaN,NaN,0.0
2,A000003,E0001,2026-08-05,Present,07:45,17:39,9.9
3,A000004,E0001,2026-08-06,Late,09:55,17:41,7.77
4,A000005,E0001,2026-08-07,Present,07:59,17:41,9.7


### 2.2 Status values

In [9]:
valid_status = {"Present", "Late", "Absent", "Leave"}
bad_status = att[~att["status"].isin(valid_status)]
print("Rows with an invalid status:", len(bad_status))
found["attendance"]["invalid_status_values"] = len(bad_status)
att["status"].value_counts(dropna=False)

Rows with an invalid status: 40


status
Present     11122
Late          884
Absent        638
Leave         356
Prsent          9
present         9
N/A             6
Pressent        6
Lte             6
ABSENT          4
Name: count, dtype: int64

### 2.3 Hours worked (valid range 0 to 24)

In [10]:
hours = pd.to_numeric(att["hours_worked"], errors="coerce")
bad_hours = att[hours.notna() & ~hours.between(0, 24)]
print("Rows with hours outside 0-24:", len(bad_hours))
print("Rows where hours is not a number:", int((hours.isna() & att["hours_worked"].notna()).sum()))
found["attendance"]["hours_worked_out_of_range"] = len(bad_hours)
bad_hours[["attendance_id", "employee_id", "status", "hours_worked"]].head(15)

Rows with hours outside 0-24: 12
Rows where hours is not a number: 0


,attendance_id,employee_id,status,hours_worked
1180,A001177,E0019,Present,25
2109,A002102,E0033,Present,30
2989,A002980,E0046,Present,48
3731,A003721,E0058,Present,-3
4518,A004504,E0070,Present,99
5011,A004997,E0077,Present,30
5465,A005451,E0084,Present,48
6696,A006676,E0103,Present,-3
8967,A008939,E0138,Late,30
9511,A009483,E0146,Present,25


### 2.4 Dates

In [11]:
iso, dmy = parse_dates(att["attendance_date"])
non_iso = iso.isna() & dmy.notna()
impossible = iso.isna() & dmy.isna()
print("Other format (dd/mm/yyyy):", int(non_iso.sum()))
print("Impossible / unreadable  :", int(impossible.sum()))
found["attendance"]["non_iso_date_format"] = int(non_iso.sum())
found["attendance"]["impossible_dates"] = int(impossible.sum())
att.loc[impossible, ["attendance_id", "employee_id", "attendance_date"]]

Other format (dd/mm/yyyy): 50
Impossible / unreadable  : 10


,attendance_id,employee_id,attendance_date
180,A000180,E0003,not_a_date
3175,A003166,E0049,not_a_date
3943,A003933,E0061,not_a_date
6226,A006208,E0096,2026-13-45
7329,A007308,E0113,not_a_date
8045,A008022,E0124,2026-13-45
8634,A008609,E0133,not_a_date
9353,A009325,E0144,31/02/2026
10270,A010238,E0158,2026-00-10
11068,A011034,E0170,31/02/2026


### 2.5 Check-in and check-out times

In [12]:
t_in = pd.to_datetime(att["check_in"], format="%H:%M", errors="coerce")
t_out = pd.to_datetime(att["check_out"], format="%H:%M", errors="coerce")

backwards = t_out < t_in
worked = att["status"].isin({"Present", "Late"})
missing_in = worked & att["check_in"].isna()

print("check_out earlier than check_in         :", int(backwards.sum()))
print("Present/Late day with no check_in       :", int(missing_in.sum()))
found["attendance"]["check_out_before_check_in"] = int(backwards.sum())
found["attendance"]["missing_check_in_on_worked_day"] = int(missing_in.sum())
att.loc[backwards, ["attendance_id", "employee_id", "check_in", "check_out", "hours_worked"]].head(10)

check_out earlier than check_in         : 25
Present/Late day with no check_in       : 31


,attendance_id,employee_id,check_in,check_out,hours_worked
847,A000845,E0013,17:25,07:36,9.82
852,A000850,E0014,17:42,07:30,10.2
1785,A001778,E0028,17:16,07:55,9.35
2315,A002308,E0036,17:38,07:59,9.65
2504,A002497,E0039,17:08,07:35,9.55
2957,A002948,E0046,17:42,07:59,9.72
3139,A003130,E0049,17:32,07:34,9.97
3141,A003132,E0049,17:14,07:56,9.3
4460,A004446,E0069,18:38,08:39,9.98
5735,A005719,E0088,17:22,07:57,9.42


### 2.6 Employees that do not exist

In [13]:
orphans = att[~att["employee_id"].isin(set(emp["employee_id"]))]
print("Attendance rows with an unknown employee_id:", len(orphans))
found["attendance"]["orphan_employee_ids"] = len(orphans)
orphans["employee_id"].value_counts().head(10)

Attendance rows with an unknown employee_id: 30


employee_id
E9080    2
E9066    2
E9045    2
E9007    1
E9088    1
E9038    1
E9052    1
E9006    1
E9082    1
E9020    1
Name: count, dtype: int64

## 3. Performance (one row = one review of one employee)

### 3.1 Structure, missing values, duplicates

In [14]:
print("Rows and columns:", perf.shape)
print()
print("Missing values per column:")
print(perf.isna().sum())
print()
dup_rows = int(perf.duplicated().sum())
print("Fully identical rows:", dup_rows)
found["performance"]["exact_duplicate_rows"] = dup_rows
perf.head()

Rows and columns: (410, 5)

Missing values per column:
employee_id        0
review_period      0
review_date        0
score              9
reviewer_status    0
dtype: int64

Fully identical rows: 10


,employee_id,review_period,review_date,score,reviewer_status
0,E0001,2025-H2,2025-12-04,4,Pending
1,E0001,2026-H1,2026-06-29,3,Approved
2,E0002,2025-H2,2025-12-21,3,Approved
3,E0002,2026-H1,2026-06-26,3,Approved
4,E0003,2025-H2,2025-12-15,5,Approved


### 3.2 Score (valid range 1 to 5)

In [15]:
score = pd.to_numeric(perf["score"], errors="coerce")
bad_score = perf[score.notna() & ~score.between(1, 5)]
missing_score = int(perf["score"].isna().sum())
print("Scores outside 1-5:", len(bad_score))
print("Missing scores    :", missing_score)
found["performance"]["score_out_of_range"] = len(bad_score)
found["performance"]["missing_score"] = missing_score
perf["score"].value_counts(dropna=False).sort_index()

Scores outside 1-5: 12
Missing scores    : 9


score
-1       4
0        2
1       15
10       2
2       30
3      164
4      128
5       52
7        4
NaN      9
Name: count, dtype: int64

### 3.3 Reviewer status

In [16]:
valid_rs = {"Approved", "Draft", "Pending"}
bad_rs = perf[~perf["reviewer_status"].isin(valid_rs)]
print("Rows with a non-standard reviewer_status:", len(bad_rs))
found["performance"]["inconsistent_status_casing"] = len(bad_rs)
perf["reviewer_status"].value_counts(dropna=False)

Rows with a non-standard reviewer_status: 15


reviewer_status
Approved     317
Draft         41
Pending       37
Approved       7
approved       5
APPROVED       3
Name: count, dtype: int64

### 3.4 Employees that do not exist

In [17]:
orphans_p = perf[~perf["employee_id"].isin(set(emp["employee_id"]))]
print("Performance rows with an unknown employee_id:", len(orphans_p))
found["performance"]["orphan_employee_ids"] = len(orphans_p)

Performance rows with an unknown employee_id: 8


## 4. Summary: our counts vs the answer key

A small difference is normal: the duplicate rows were copied from other rows, so a copy of a damaged row counts as another damaged row. A **big** difference means something to investigate.

In [18]:
rows = []
for dataset, counts in found.items():
    for problem, n_found in counts.items():
        n_key = key[dataset].get(problem)
        rows.append({"file": dataset, "problem": problem, "found": n_found,
                     "answer_key": n_key, "difference": n_found - n_key})
summary = pd.DataFrame(rows)
summary

,file,problem,found,answer_key,difference
0,employees,missing_department,5,5,0
1,employees,missing_job_title,4,4,0
2,employees,duplicate_employee_id_rows,6,6,0
3,employees,inconsistent_department_labels,25,24,1
4,employees,job_title_casing_issues,8,8,0
5,employees,non_iso_hire_date_format,11,10,1
6,employees,future_hire_date,3,3,0
7,attendance,exact_duplicate_rows,40,40,0
8,attendance,invalid_status_values,40,40,0
9,attendance,hours_worked_out_of_range,12,12,0
